# Immunogenicity tool test: Testing setttings for tools: NetMHC_II_pan_EL
Author: Rebecka Antonsson\
Version: 1 (02-03-2026)

# Notebook explanation
Notebook to calculate the scores of the outputs from the tool BioPhi(OASis) with different settings.

Settings tested are:\
\
netMHC_II_pan_EL_peplength15_1allels_wind5
netMHC_II_pan_EL_peplength15_27allels_wind5
netMHC_II_pan_EL_peplength15_7allels_wind5

netMHC_II_pan_EL_peplength12_27allels_wind5
netMHC_II_pan_EL_peplength12_7allels_wind5

netMHC_II_pan_EL_peplength15_27allels_wind2
netMHC_II_pan_EL_peplength15_7allels_wind2

netMHC_II_pan_EL_peplength15_27allels_wind10


The only difference is the sliding windoww lengtht that is used when testing each sequence. There are seven different window sizes called peptidelength.


netmhcpan_el percentile:The percentile rank generated by comparing the peptide's IC50/score against those of a set of random peptides from SWISSPROT database.  Probability that the peptide will be natrually present on MHC-1.
IEDB themself recomend using a threshold of <=1% to cover most of the immune response
Here the score is calculated as nr of rows with netMHCpan percentile <=1, divided by total number of rows. Meaning that higher number indicated higher immunogenicity. The score can be described as the percentage of MHC-peptides that have a netMHCpan percentile below 1

For each tool_output with different settings a ranking from 1 to 37 will be done, each antibody will hence be given a number 1 to 37
This ranking will be compared to the known/ clinically determined immunigenecity ranking of the antibodies. The clinical immunigenecity ranking is based on anti-drug antibody (ADA) data. The two nanobodies will be held outside the ranking since they have risk of behaving differently, and special intreset is taken in how the tool perform at prediction those.

For each setting three measurments on how good they perform will be calculated:
    1. Mean absolute rank error (MARE) for antibodies
    2. Spearman rank correlation
    3. Mean absolute rank error for the two nanobodies

1. MARE is just the absolute difference between the known rank of the antibody and the predicted rank
2. Spearman rank correlation is a statistical test that can compare two lists of ranking and tell how well they align. 1 is a perfect correlation
0 means no relation between the two variables (no correlation, random) and -1 is a perfect reversed correlation (very bad).
3. Same calculation as for 1, but separated from the rest. Here a separate ranking where the 2 nanobodies are included are performed, hence antobodies and nanobodies get a number 1 to 39 and the MARE for the nanobodies is calculated. 

In [1]:
import pandas as pd
from scipy.stats import spearmanr

In [2]:
# Load the tool outputs with the different setttings, store each in a separate pandas DataFrame
netMHC_II_pan_EL_peplength15_1allels_wind5 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_1allels_wind5_02_03_2026.csv")
netMHC_II_pan_EL_peplength15_27allels_wind5 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_27allels_wind5_02_03_2026.csv")
netMHC_II_pan_EL_peplength15_7allels_wind5 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_7allels_wind5_02_03_2026.csv")

netMHC_II_pan_EL_peplength12_27allels_wind5 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength12_27allels_wind5_02_03_2026.csv")
netMHC_II_pan_EL_peplength12_7allels_wind5 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength12_7allels_wind5_02_03_2026.csv")

netMHC_II_pan_EL_peplength15_27allels_wind2 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_27allels_wind2_02_03_2026.csv")
netMHC_II_pan_EL_peplength15_7allels_wind2 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_7allels_wind2_02_03_2026.csv")

netMHC_II_pan_EL_peplength15_27allels_wind10 = pd.read_csv("tool_outputs/netMHC_II_pan_EL_peplength15_27allels_wind10_02_03_2026.csv")


# Then also load the sequence table for each tool output. 
# This is because in the tool output of the scores each antibody sequence has just been given an number (1-39)
# With the sequence table I can map the antibody name back to the number it was given
seq_table_netMHC_II_pan_EL_peplength15_1allels_wind5 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_1allels_wind5_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength15_27allels_wind5 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_27allels_wind5_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength15_7allels_wind5 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_7allels_wind5_02_03_2026.csv")  
seq_table_netMHC_II_pan_EL_peplength12_27allels_wind5 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength12_27allels_wind5_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength12_7allels_wind = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength12_7allels_wind5_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength15_27allels_wind2 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_27allels_wind2_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength15_7allels_wind2 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_7allels_wind2_02_03_2026.csv")
seq_table_netMHC_II_pan_EL_peplength15_27allels_wind10 = pd.read_csv("tool_outputs/seq_table_netMHC_II_pan_EL_peplength15_27allels_wind10_02_03_2026.csv")


In [3]:
# Create dictionary with all df names too be able to loop through them
all_dfs = {
    'netMHC_II_pan_EL_peplength15_1allels_wind5': netMHC_II_pan_EL_peplength15_1allels_wind5,
    'netMHC_II_pan_EL_peplength15_27allels_wind5': netMHC_II_pan_EL_peplength15_27allels_wind5,
    'netMHC_II_pan_EL_peplength15_7allels_wind5': netMHC_II_pan_EL_peplength15_7allels_wind5,
    'netMHC_II_pan_EL_peplength12_27allels_wind5': netMHC_II_pan_EL_peplength12_27allels_wind5,
    'netMHC_II_pan_EL_peplength12_7allels_wind5': netMHC_II_pan_EL_peplength12_7allels_wind5,
    'netMHC_II_pan_EL_peplength15_27allels_wind2': netMHC_II_pan_EL_peplength15_27allels_wind2,
    'netMHC_II_pan_EL_peplength15_7allels_wind2': netMHC_II_pan_EL_peplength15_7allels_wind2,
    'netMHC_II_pan_EL_peplength15_27allels_wind10': netMHC_II_pan_EL_peplength15_27allels_wind10
}

seq_tables = {
    'netMHC_II_pan_EL_peplength15_1allels_wind5': seq_table_netMHC_II_pan_EL_peplength15_1allels_wind5,
    'netMHC_II_pan_EL_peplength15_27allels_wind5': seq_table_netMHC_II_pan_EL_peplength15_27allels_wind5,
    'netMHC_II_pan_EL_peplength15_7allels_wind5': seq_table_netMHC_II_pan_EL_peplength15_7allels_wind5,
    'netMHC_II_pan_EL_peplength12_27allels_wind5': seq_table_netMHC_II_pan_EL_peplength12_27allels_wind5,
    'netMHC_II_pan_EL_peplength12_7allels_wind5': seq_table_netMHC_II_pan_EL_peplength12_7allels_wind,
    'netMHC_II_pan_EL_peplength15_27allels_wind2': seq_table_netMHC_II_pan_EL_peplength15_27allels_wind2,
    'netMHC_II_pan_EL_peplength15_7allels_wind2': seq_table_netMHC_II_pan_EL_peplength15_7allels_wind2,
    'netMHC_II_pan_EL_peplength15_27allels_wind10': seq_table_netMHC_II_pan_EL_peplength15_27allels_wind10
}

In [7]:
# Sanity check
# Check so that all dataframes have all the 39 antibodies and the 27 alleles
for name, df in all_dfs.items():
    # check if unique values in seq # column is 39)
    if df['seq #'].nunique()!=39:
        print(f'{name, "does not have 39 antibodies/nanobodies"}')
    else:
        # check if number of unique valies in HLA allele column is 27)
        if df['allele'].nunique()==27:
            print(f'{name,"has 27 HLA alleles"}')
        elif df['allele'].nunique()==7:
            print(f'{name,"has 7 HLA alleles"}')
        elif df['allele'].nunique()==1:
            print(f'{name,"has 1 HLA allele"}')
        else:
            print(f'{name,"does not have 27 or 7 HLA alleles"}')


('netMHC_II_pan_EL_peplength15_1allels_wind5', 'has 1 HLA allele')
('netMHC_II_pan_EL_peplength15_27allels_wind5', 'has 27 HLA alleles')
('netMHC_II_pan_EL_peplength15_7allels_wind5', 'has 7 HLA alleles')
('netMHC_II_pan_EL_peplength12_27allels_wind5', 'has 27 HLA alleles')
('netMHC_II_pan_EL_peplength12_7allels_wind5', 'has 7 HLA alleles')
('netMHC_II_pan_EL_peplength15_27allels_wind2', 'has 27 HLA alleles')
('netMHC_II_pan_EL_peplength15_7allels_wind2', 'has 7 HLA alleles')
('netMHC_II_pan_EL_peplength15_27allels_wind10', 'has 27 HLA alleles')


In [ ]:
# Clean dataframes a bit, and instert the correct antibody names
for key in all_dfs:
    df = all_dfs[key]
    seq_table = seq_tables[key] 

    # Insert the sequence names (Antibody names) into the dataframe with scores, map by seq #, which exists in both dataframes
    df = df.merge(seq_table[['seq #', 'sequence name']], how='left')

    # Rename the column "sequence name" to "Antibody", 
    df.rename(columns={'sequence name': 'Antibody'}, inplace=True)

    # remove the two columns called seq # from the calc_rank_table
    df = df.drop(columns=['seq #'])

    # Place the anotbody columns as the first column beacuse its easier to read

    # Remove the last column and save it as a variable
    last_col = df.pop(df.columns[-1])
    # Insert the column as the first column
    df.insert(0, last_col.name, last_col)

    all_dfs[key] = df